In [1]:
import os
import sys
import json
from tqdm import tqdm
import shutil
import numpy as np
import pandas as pd
import cv2 as cv
import csv

sys.path.insert(0, "../../packages/python")
from models import cell_segmentation as segmentators

2025-07-26 23:46:08.846501: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-26 23:46:08.852675: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753584368.859689   49788 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753584368.861789   49788 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753584368.867401   49788 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
IMG_TARGET_SIDE = 225

CROPS_PATH = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/dividing/'

IMAGES_PATH_1 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/ina/images/'
IMAGES_PATH_2 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/onion_cell_merged/images/train/'
IMAGES_PATH_3 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/onion_cell_merged/images/test/'
IMAGES_PATH_4 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/onion_cell_merged/images/valid/'

CSV_PATH_1 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/ina/data/'
CSV_PATH_2 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/onion_cell_merged/images/train/'
CSV_PATH_3 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/onion_cell_merged/data_v2/test/'
CSV_PATH_4 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/onion_cell_merged/data_v2/valid/'

JSON_PATH = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/temp/datasets_area_data.json'

OUTPUT_PATH = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/dividing_enlarged'
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [3]:
crops = [CROPS_PATH + crops_path for crops_path in sorted(os.listdir(CROPS_PATH))]

csvs_1 = [CSV_PATH_1 + csvs_path for csvs_path in sorted(os.listdir(CSV_PATH_1))]
csvs_2 = [CSV_PATH_2 + csvs_path for csvs_path in sorted(os.listdir(CSV_PATH_2))]
csvs_3 = [CSV_PATH_3 + csvs_path for csvs_path in sorted(os.listdir(CSV_PATH_3))]
csvs_4 = [CSV_PATH_4 + csvs_path for csvs_path in sorted(os.listdir(CSV_PATH_4))]
csvs = csvs_1 + csvs_2 + csvs_3 + csvs_4

images_1 = [IMAGES_PATH_1 + images_path for images_path in sorted(os.listdir(IMAGES_PATH_1))]
images_2 = [IMAGES_PATH_2 + images_path for images_path in sorted(os.listdir(IMAGES_PATH_2))]
images_3 = [IMAGES_PATH_3 + images_path for images_path in sorted(os.listdir(IMAGES_PATH_3))]
images_4 = [IMAGES_PATH_4 + images_path for images_path in sorted(os.listdir(IMAGES_PATH_4))]
images = images_1 + images_2 + images_3 + images_4

with open(JSON_PATH, 'r') as f: #json with the information of the filename of the images
    area_data = json.load(f)

In [ ]:
for crop_path in crops:
    # 1. Extract IDs from crop name
    crop_filename = os.path.basename(crop_path)
    subsection, image_name, cell_id = crop_filename.split('_')
    cell_id = cell_id.split('.')[0]  # Remove extension if present

    # 2. Find the corresponding image
    image_filename = f"{subsection}_{image_name}"
    image_path = None
    for img_path in images:
        if image_filename in img_path:
            image_path = img_path
            break

    if not image_path:
        print(f"Warning: Image not found for crop {crop_filename}")
        continue

    # 3. Find the corresponding CSV
    csv_filename = f"{subsection}_{image_name}.csv"
    csv_path = None
    for csv_file in csvs:
        if csv_filename in csv_file:
            csv_path = csv_file
            break

    if not csv_path:
        print(f"Warning: CSV not found for crop {crop_filename}")
        continue

    # 4. Read the CSV and extract cell data
    try:
        df = pd.read_csv(csv_path)
        df_bbox = df[df['cell_id'] == int(cell_id)]  # Pandas reads cell_id as int
        if not df_bbox.empty:
            print(f"Crop: {crop_filename}, Image: {image_filename}, Cell Data:\n{df_bbox.to_string()}")

            resize_factor = IMG_TARGET_SIDE/area_data['INA']['lado_cuadrado']

            image_group = image_name[0] if image_name[0].isalpha() else "INA" 
            image_side = area_data[image_group]['lado_cuadrado']
            image_resize_factor = 200#IMG_TARGET_SIDE#int(resize_factor * image_side)

            img = cv.imread(image_path)
            df = pd.read_csv(csv_path)

            for _, row in df_bbox.iterrows():
                # cell_id = row['cell_id']    
                x, y, w, h = row['x'], row['y'], row['w'], row['h']
                x, y, w, h = segmentators.CellMaskGenerator.adjust_bbox(segmentators.CellMaskGenerator, x, y, w, h, image_resize_factor*image_resize_factor, img.shape[0], img.shape[1])

                crop = cv.resize(img[y:y+h, x:x+w], (IMG_TARGET_SIDE, IMG_TARGET_SIDE))
                output_path = os.path.join(OUTPUT_PATH, crop_filename)
                cv.imwrite(output_path, crop)
        break
        # else:
        #     print(f"Warning: Cell ID {cell_id} not found in CSV {csv_filename}")
    except Exception as e:
        print(f"An error occurred while reading CSV: {e}")


Crop: 001_00001_10.png, Image: 001_00001, Cell Data:
    Unnamed: 0   area    x     y    w    h  bbox_area      image  cell_id
10          10  24289  534  1088  202  246      49692  001_00001       10
An error occurred while reading CSV: OpenCV(4.10.0) /io/opencv/modules/imgproc/src/resize.cpp:4152: error: (-215:Assertion failed) !ssize.empty() in function 'resize'

Crop: 001_00001_7.png, Image: 001_00001, Cell Data:
   Unnamed: 0   area    x     y    w    h  bbox_area      image  cell_id
7           7  24638  318  1385  208  250      52000  001_00001        7
An error occurred while reading CSV: OpenCV(4.10.0) /io/opencv/modules/imgproc/src/resize.cpp:4152: error: (-215:Assertion failed) !ssize.empty() in function 'resize'

Crop: 001_00002_1.png, Image: 001_00002, Cell Data:
   Unnamed: 0   area    x    y    w    h  bbox_area      image  cell_id
1           1  64082  198  424  264  357      94248  001_00002        1
An error occurred while reading CSV: OpenCV(4.10.0) /io/opencv/module